In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client()

client

/g/data/xp65/public/apps/med_conda/envs/access-med-0.11/lib/python3.11/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42303 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/42303/status,
Dashboard: /proxy/42303/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42329,Workers: 7
Dashboard: /proxy/42303/status,Total threads: 14
Started: Just now,Total memory: 63.00 GiB
Comm: tcp://127.0.0.1:35971,Total threads: 2
Dashboard: /proxy/41127/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:38595,


In [2]:
#import all the stuff
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib
#import xclim.indices
from datetime import timedelta
import cartopy.crs as ccrs
import cartopy as cart
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import glob

In [3]:
#Set parameters
CMIP='CMIP6'
#AGENCY = 'CSIRO' 
#RCM = 'CCAM-v2203-SN'
AGENCY = 'BOM' 
RCM = 'BARPA-R'

#GCM = 'NorESM2-MM' #ensemble = 'r1i1p1f1' #Done
GCM = 'ACCESS-ESM1-5' 
ensemble = 'r6i1p1f1' #Done
#GCM = 'ACCESS-CM2' #ensemble = 'r4i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' #ensemble = 'r1i1p1f2' #CSIRO Done, no BOM
#GCM = 'MPI-ESM1-2-HR' #ensemble = 'r1i1p1f1' #BOM done, no CSIRO
#GCM = 'CESM2' #ensemble = 'r11i1p1f1' #Done
#GCM = 'CMCC-ESM2' #ensemble = 'r1i1p1f1' #Done
#GCM = 'EC-Earth3' #ensemble = 'r1i1p1f1' #Done

#pathway = 'ssp126'
pathway = 'ssp370'

output_dir = '/g/data/ia39/ncra/bushfire/vpd/threshold18'

In [4]:
#GWLs = ['1.2', '1.5', '2.0', '3.0']
#halves = ['1st', '2nd']

In [5]:
#Create half files with the sum of the number of days per month with vpd greater than 18
GWLs = ['1.2', '1.5']
halves = ['1st', '2nd']

for chosen_gwl in GWLs:
    print(chosen_gwl)
    infile=f"/g/data/ia39/ncra/bushfire/vpd/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{pathway}_{GCM}_{RCM}_gwl{chosen_gwl}_vpd.nc"
    vpd_ds = xr.open_dataset(infile)['vpd']
    print('infile opened')
    
    for half in halves:
        print(half)
        if half == '1st':
            vpd_days_above_18 = vpd_ds[0:2737] > 18
        else: 
            vpd_days_above_18 = vpd_ds[2737:] > 18
        
        print('half thresholding done')
        monthly_count_vpd_days_above_18 = vpd_days_above_18.groupby('time.month').sum('time')
        print('half monthly count done')

        file_name_vpd = half + '_' + pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd_days_gt_18.nc' 
        output_file_location = output_dir + '/half_files/' + file_name_vpd
        monthly_count_vpd_days_above_18.to_netcdf(output_file_location, engine='netcdf4')
        print(output_file_location)
    
        vpd_ds.close()


1.2
infile opened
1st
half thresholding done
half monthly count done
/g/data/ia39/ncra/bushfire/vpd/threshold18/half_files/1st_ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.2_vpd_days_gt_18.nc
2nd
half thresholding done
half monthly count done
/g/data/ia39/ncra/bushfire/vpd/threshold18/half_files/2nd_ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.2_vpd_days_gt_18.nc
1.5
infile opened
1st
half thresholding done
half monthly count done
/g/data/ia39/ncra/bushfire/vpd/threshold18/half_files/1st_ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.5_vpd_days_gt_18.nc
2nd
half thresholding done
half monthly count done
/g/data/ia39/ncra/bushfire/vpd/threshold18/half_files/2nd_ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.5_vpd_days_gt_18.nc


In [6]:
for chosen_gwl in GWLs:
    a_half = output_dir + '/half_files/1st_' + pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd_days_gt_18.nc'
    b_half = output_dir + '/half_files/2nd_' + pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd_days_gt_18.nc'

    full_sum = xr.open_dataset(a_half)['vpd'] + xr.open_dataset(b_half)['vpd']

    output_file_location = output_dir + '/' + pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd_days_gt_18.nc'
    full_sum.to_netcdf(output_file_location)
    print(output_file_location)


/g/data/ia39/ncra/bushfire/vpd/threshold18/ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.2_vpd_days_gt_18.nc
/g/data/ia39/ncra/bushfire/vpd/threshold18/ssp370_ACCESS-ESM1-5_BARPA-R_gwl1.5_vpd_days_gt_18.nc
